# Inteligencia Artificial Avanzada

## Entrega 3 - Experimento 4B: Object Detection + CNN GAP con tasa adaptable

Detector SSD MobileNet COCO, umbral 0,25, margen 10 % e imagen completa si no detecta. CNN GAP desde cero con Adam 0,001, ReduceLROnPlateau (factor 0,5, paciencia 4, minimo 0,00001) y Early Stopping con paciencia 12. No hay augmentation ni Transfer Learning en el clasificador.

**Evidencia actual:** `run_20260908_013049_exp4_img224_bs32_lr0.001`, disponible en `../resultados/04B_ObjectDetection_CNN_GAP_LR/`. Accuracy de especie 57,625 %, macro-F1 57,601 %, accuracy regional 77,375 %; 100 epocas ejecutadas y mejor epoca 88. Estas cifras provienen de los CSV y JSON, no de una nueva ejecucion local del notebook.

Se retiraron las salidas de la corrida anterior (86 epocas y 55,875 %) para no mezclarlas con la exportacion actual. El codigo queda sin ejecutar; las curvas, matrices y predicciones actuales estan en la carpeta de resultados.

La seleccion y particiones siguen siendo diferentes de 4/4C/5: 2.967 observaciones compartidas y 1.645 reasignadas. No permite aislar el efecto de la tasa de aprendizaje. Las imagenes preparadas quedan temporalmente en Colab y los resultados se exportan una sola vez a `runs`.


In [ ]:
#@title 1. Configuracion general

EXPERIMENT_NUMBER = 4
EXPERIMENT_ID = "exp4b_object_detection_crop_cnn_gap_lr"
EXPERIMENT_TITLE = "Exp. 4B - Object Detection + CNN GAP con LR adaptable"

SEED = 42 #@param {type:"integer"}
MAX_IMAGES_PER_SPECIES = 400 #@param {type:"integer"}
MIN_IMAGES_RECOMMENDED = 100 #@param {type:"integer"}

IMG_SIZE = 224 #@param {type:"integer"}
BATCH_SIZE = 32 #@param {type:"integer"}
EPOCHS = 200 #@param {type:"integer"}
LEARNING_RATE = 0.001 #@param {type:"number"}

TEST_FRACTION = 0.25
VAL_FRACTION_TOTAL = 0.15
TRAIN_FRACTION_TOTAL = 0.60

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
FORCE_REDOWNLOAD = False #@param {type:"boolean"}

# La condicion nativa/exotica se determina por catalogo regional.
RESTRICT_OBSERVATIONS_TO_BA = False #@param {type:"boolean"}

# Detector preentrenado MobileNet SSD. En COCO, la clase 64 corresponde a potted plant.
OBJECT_DETECTION_MODEL_URL = "https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2"
DETECTION_CLASS_IDS = [64]
DETECTION_CLASS_NAMES = {64: "potted plant"}
DETECTION_SCORE_THRESHOLD = 0.25 #@param {type:"number"}
CROP_PADDING = 0.10 #@param {type:"number"}
FALLBACK_CENTER_CROP = False #@param {type:"boolean"}

print("Experimento:", EXPERIMENT_NUMBER, "-", EXPERIMENT_TITLE)
print(f"Split objetivo: train={TRAIN_FRACTION_TOTAL:.0%}, val={VAL_FRACTION_TOTAL:.0%}, test={TEST_FRACTION:.0%}")


In [ ]:
#@title 2. Librerias y reproducibilidad

import os
import re
import io
import json
import time
import shutil
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import requests
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    accuracy_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    import tensorflow_hub as hub
except ImportError:
    !pip -q install tensorflow-hub
    import tensorflow_hub as hub

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)



def slugify_species(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(name).strip()).strip("_")

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

def dataset_partition_sha256(frame):
    import hashlib
    columns = ["observation_id", "photo_id", "scientific_name", "condition_ba", "split"]
    data = frame[columns].astype(str).copy()
    data["scientific_name"] = data["scientific_name"].map(
        lambda value: re.sub(r"[^a-z0-9]+", "_", value.strip().casefold()).strip("_")
    )
    records = sorted(data.values.tolist())
    payload = json.dumps(records, ensure_ascii=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


In [ ]:
#@title 3. Acceder a Google Drive y preparar directorios

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=False)
    BASE_DIR = Path('/content/gdrive/MyDrive/2026_IAA_GrupoC')
else:
    BASE_DIR = Path('/content/2026_IAA_GrupoC')


RAW_DIR = BASE_DIR / "dataset_raw"
MANIFEST_PATH = BASE_DIR / "manifest_inaturalist.csv"

ARTIFACTS_ROOT = BASE_DIR / "artifacts_entrega3"
RESULTS_ROOT = ARTIFACTS_ROOT / "runs"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_NAME = f"run_{RUN_TIMESTAMP}_{EXPERIMENT_ID}"
RUN_DIR = RESULTS_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
ARTIFACTS_DIR = RUN_DIR

WORK_DIR = Path("/content/iaa_work") / RUN_NAME
WORK_DIR.mkdir(parents=True, exist_ok=False)
CROP_SPLIT_DIR = WORK_DIR / "dataset_recortado"

BALANCED_MANIFEST_PATH = RUN_DIR / "manifest_balanceado.csv"
SPLIT_MANIFEST_PATH = RUN_DIR / "manifest_splits.csv"
CROP_MANIFEST_PATH = RUN_DIR / "manifest_splits_object_detection_crop.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
print("Resultados en Drive:", RUN_DIR)
print("Imagenes temporales en Colab:", WORK_DIR)


## 4. Catálogo inicial de especies

Para la prueba de concepto se trabaja con **8 clases visuales**. La lista está pensada para ser suficientemente acotada para la Entrega 2 y, al mismo tiempo, permitir evaluar un problema multiclase real.


In [ ]:
#@title 4.1 Catálogo de especies del prototipo

SPECIES_CATALOG = pd.DataFrame([

    {
        "scientific_name": "Erythrina crista-galli",
        "common_name": "Ceibo",
        "condition_ba": "Nativa",
        "status_source": "Flora Argentina",

    },

    {
        "scientific_name": "Passiflora caerulea",
        "common_name": "Mburucuyá",
        "condition_ba": "Nativa",
        "status_source": "Flora Argentina",

    },

    {
        "scientific_name": "Cortaderia selloana",
        "common_name": "Cortadera",
        "condition_ba": "Nativa",
        "status_source": "Flora Argentina",

    },

    {
        "scientific_name": "Phytolacca dioica",
        "common_name": "Ombú",
        "condition_ba": "Nativa",
        "status_source": "Flora Argentina",

    },

    {
        "scientific_name": "Ligustrum lucidum",
        "common_name": "Ligustro",
        "condition_ba": "Exótica",
        "status_source": "Flora Argentina",

    },

    {
        "scientific_name": "Gleditsia triacanthos",
        "common_name": "Acacia negra",
        "condition_ba": "Exótica",
        "status_source": "Flora Argentina",

    },

    {
        "scientific_name": "Melia azedarach",
        "common_name": "Paraíso",
        "condition_ba": "Exótica",
        "status_source": "Flora Argentina",

    },

    {
        "scientific_name": "Iris pseudacorus",
        "common_name": "Lirio amarillo",
        "condition_ba": "Exótica",
        "status_source": "Flora Argentina",

    },

])

SPECIES_CATALOG["folder_name"] = (
    SPECIES_CATALOG["scientific_name"]
    .apply(slugify_species)
)

display(SPECIES_CATALOG)
print("\nCantidad de especies:", len(SPECIES_CATALOG))
print(SPECIES_CATALOG["condition_ba"].value_counts())


In [ ]:
#@title 4.2 Validacion del catalogo regional

required_fields = [
    "scientific_name",
    "common_name",
    "condition_ba",
    "status_source",
    "folder_name"
]

missing_catalog_data = SPECIES_CATALOG[required_fields].isna().sum()

print("Campos faltantes:")
display(missing_catalog_data.to_frame("faltantes"))

print("\nCondiciones:")
display(
    SPECIES_CATALOG["condition_ba"]
    .value_counts()
    .rename("cantidad")
    .to_frame()
)

assert SPECIES_CATALOG["folder_name"].is_unique, "Los folder_name del catalogo deben ser unicos."


## 5. Construcción del dataset desde iNaturalist

Se descargará **una sola fotografía por observación** para reducir el riesgo de tener tomas casi idénticas del mismo registro repartidas entre entrenamiento y prueba.

También se conservará un **manifest CSV** con:
- ID de observación;
- especie;
- condición regional;
- URL de la observación;
- URL de la foto;
- licencia;
- atribución;
- fecha y coordenadas cuando estén disponibles;
- ruta local.

Para el TP se filtran fotografías con licencias Creative Commons compatibles con uso académico y transformación de la imagen.


In [ ]:
#@title 5.1 Funciones para consultar iNaturalist y descargar imágenes

INAT_API = "https://api.inaturalist.org/v1"
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "UTN-FRBA-IAA-TP-Plantas/1.0 (academic prototype)"
})

# Licencias de fotografía que admitimos en este prototipo.
ALLOWED_LICENSES = {
    "cc0",
    "cc-by",
    "cc-by-sa",
    "cc-by-nc",
    "cc-by-nc-sa",
}

# Bounding box aproximado de Provincia de Buenos Aires.
BA_BBOX = {
    "swlat": -41.2,
    "swlng": -63.6,
    "nelat": -33.2,
    "nelng": -56.4,
}

def resolve_taxon_id(scientific_name):
    """Busca el taxon_id de una especie en iNaturalist."""
    r = SESSION.get(
        f"{INAT_API}/taxa",
        params={"q": scientific_name, "rank": "species", "per_page": 10},
        timeout=30
    )
    r.raise_for_status()
    results = r.json().get("results", [])

    # Primero intenta coincidencia exacta.
    for taxon in results:
        if taxon.get("name", "").lower() == scientific_name.lower():
            return taxon["id"]

    # Si no existe coincidencia exacta, toma el primer resultado.
    if results:
        warnings.warn(
            f"No hubo coincidencia exacta para {scientific_name}. "
            f"Se utilizará: {results[0].get('name')}"
        )
        return results[0]["id"]

    raise ValueError(f"No se encontró taxón para: {scientific_name}")


def photo_medium_url(photo):
    """
    iNaturalist suele devolver una URL con tamaño 'square'.
    La reemplazamos por 'medium' para obtener una imagen útil sin bajar archivos enormes.
    """
    url = photo.get("url")
    if not url:
        return None
    return url.replace("/square.", "/medium.").replace("square.", "medium.")


def iter_observation_records(scientific_name, taxon_id, condition_ba, max_records):
    """Devuelve registros candidatos, uno por observación."""
    page = 1
    collected = 0
    seen_obs = set()

    while collected < max_records:
        params = {
            "taxon_id": taxon_id,
            "quality_grade": "research",
            "photos": "true",
            "per_page": 200,
            "page": page,
            "order_by": "created_at",
            "order": "desc",
        }

        if RESTRICT_OBSERVATIONS_TO_BA:
            params.update(BA_BBOX)

        r = SESSION.get(f"{INAT_API}/observations", params=params, timeout=45)
        r.raise_for_status()
        payload = r.json()
        results = payload.get("results", [])

        if not results:
            break

        for obs in results:
            obs_id = obs.get("id")
            if obs_id in seen_obs:
                continue
            seen_obs.add(obs_id)

            photos = obs.get("photos") or []
            if not photos:
                continue

            # Se toma una sola foto por observación.
            photo = photos[0]
            license_code = (photo.get("license_code") or "").lower()

            if license_code not in ALLOWED_LICENSES:
                continue

            url = photo_medium_url(photo)
            if not url:
                continue

            geojson = obs.get("geojson") or {}
            coords = geojson.get("coordinates") or [None, None]
            lng = coords[0] if len(coords) > 0 else None
            lat = coords[1] if len(coords) > 1 else None

            yield {
                "observation_id": obs_id,
                "scientific_name": scientific_name,
                "condition_ba": condition_ba,
                "observation_url": f"https://www.inaturalist.org/observations/{obs_id}",
                "photo_id": photo.get("id"),
                "photo_url": url,
                "license_code": license_code,
                "attribution": photo.get("attribution"),
                "observed_on": obs.get("observed_on"),
                "latitude": lat,
                "longitude": lng,
            }

            collected += 1
            if collected >= max_records:
                break

        page += 1
        if page > 20:
            break

        time.sleep(0.15)


def download_image(url, out_path):
    """Descarga, valida y normaliza una imagen a JPEG RGB."""
    r = SESSION.get(url, timeout=45)
    r.raise_for_status()

    img = Image.open(io.BytesIO(r.content)).convert("RGB")
    if min(img.size) < 80:
        raise ValueError(f"Imagen demasiado pequeña: {img.size}")

    out_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(out_path, format="JPEG", quality=92)


def build_dataset_from_inaturalist():
    if FORCE_REDOWNLOAD and RAW_DIR.exists():
        print("FORCE_REDOWNLOAD=True -> borrando dataset_raw existente...")
        shutil.rmtree(RAW_DIR)
        RAW_DIR.mkdir(parents=True, exist_ok=True)

    if MANIFEST_PATH.exists() and not FORCE_REDOWNLOAD:
        print("Se encontró manifest existente. Se reutiliza:")
        print(MANIFEST_PATH)
        df = pd.read_csv(MANIFEST_PATH)
        return df

    records_downloaded = []

    for _, row in SPECIES_CATALOG.iterrows():
        species = row["scientific_name"]
        condition = row["condition_ba"]
        folder = RAW_DIR / row["folder_name"]

        print("\n" + "=" * 80)
        print("Especie:", species, "|", condition)

        taxon_id = resolve_taxon_id(species)
        print("taxon_id:", taxon_id)

        ok = 0
        errors = 0

        # Pedimos algo más que el objetivo porque algunas descargas pueden fallar.
        candidate_limit = int(MAX_IMAGES_PER_SPECIES * 1.5)

        for rec in iter_observation_records(
            scientific_name=species,
            taxon_id=taxon_id,
            condition_ba=condition,
            max_records=candidate_limit
        ):
            if ok >= MAX_IMAGES_PER_SPECIES:
                break

            out_path = folder / f"obs_{rec['observation_id']}_photo_{rec['photo_id']}.jpg"

            try:
                if not out_path.exists():
                    download_image(rec["photo_url"], out_path)

                rec["local_path"] = str(out_path)
                rec["folder_name"] = row["folder_name"]
                rec["common_name"] = row["common_name"]
                records_downloaded.append(rec)

                ok += 1

                if ok % 25 == 0:
                    print(f"  {ok}/{MAX_IMAGES_PER_SPECIES} imágenes válidas")

            except Exception as e:
                errors += 1
                if errors <= 3:
                    print("  Error descartado:", type(e).__name__, str(e)[:120])

        print(f"Final: {ok} imágenes válidas | {errors} errores de descarga/lectura")

    df = pd.DataFrame(records_downloaded)
    df.to_csv(MANIFEST_PATH, index=False)

    print("\nManifest guardado en:", MANIFEST_PATH)
    return df


In [ ]:
#@title 5.2 Descargar / reutilizar dataset

manifest = build_dataset_from_inaturalist()

print("\nTotal imágenes:", len(manifest))
display(
    manifest.groupby(["scientific_name", "condition_ba"])
            .size()
            .rename("cantidad")
            .reset_index()
)


In [ ]:
#@title 6. Auditar dataset: archivos, cantidades y ejemplos

# Verifica que los archivos realmente existan.
manifest["file_exists"] = manifest["local_path"].apply(lambda p: Path(p).exists())
missing = (~manifest["file_exists"]).sum()
print("Archivos faltantes:", missing)

manifest = manifest[manifest["file_exists"]].copy()

counts = manifest["scientific_name"].value_counts().sort_index()
display(counts.rename("cantidad").to_frame())

plt.figure(figsize=(11, 4))
counts.plot(kind="bar")
plt.title("Cantidad de imágenes válidas por especie")
plt.ylabel("Imágenes")
plt.xlabel("Especie")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Muestra 2 ejemplos por especie.
n_per_class = 2
sample_df = (
    manifest.groupby("scientific_name", group_keys=False)
            .apply(lambda x: x.sample(min(n_per_class, len(x)), random_state=SEED))
            .reset_index(drop=True)
)

fig, axes = plt.subplots(len(SPECIES_CATALOG), n_per_class, figsize=(8, 3 * len(SPECIES_CATALOG)))
if len(SPECIES_CATALOG) == 1:
    axes = np.array([axes])

for ax in np.array(axes).reshape(-1):
    ax.axis("off")

for i, (_, row) in enumerate(sample_df.iterrows()):
    species_idx = list(SPECIES_CATALOG["scientific_name"]).index(row["scientific_name"])
    pos_in_species = sum(sample_df.iloc[:i]["scientific_name"] == row["scientific_name"])
    ax = axes[species_idx, pos_in_species]
    img = Image.open(row["local_path"])
    ax.imshow(img)
    ax.set_title(f"{row['scientific_name']}\n{row['condition_ba']}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

if counts.min() < MIN_IMAGES_RECOMMENDED:
    warnings.warn(
        f"La clase con menos imágenes tiene {counts.min()}. "
        "Conviene revisar la lista de especies o aumentar la descarga."
    )


## 7. Balanceo y particion

Se usan 3.200 imagenes, 400 por especie, distribuidas en 1.920 de entrenamiento, 480 de validacion y 800 de prueba. La semilla 42 no garantiza las mismas particiones si cambia el dataset o su orden. Esta corrida tiene otra particion respecto del 4 y no es una ablacion controlada del learning rate. Para repetir esa comparacion se debe conservar el manifiesto original del 4 antes de preparar las imagenes.


In [ ]:
#@title 7.1 Balancear clases

counts = manifest.groupby("scientific_name").size()
n_balance = int(min(counts.min(), MAX_IMAGES_PER_SPECIES))

print("Cantidad balanceada por especie:", n_balance)

balanced = (
    manifest.groupby("scientific_name", group_keys=False)
            .apply(lambda x: x.sample(n=n_balance, random_state=SEED))
            .reset_index(drop=True)
)

balanced.to_csv(BALANCED_MANIFEST_PATH, index=False)

display(
    balanced.groupby(["scientific_name", "condition_ba"])
            .size()
            .rename("cantidad")
            .reset_index()
)

print("Total balanceado:", len(balanced))


In [ ]:
#@title 7.2 Separar 60% entrenamiento / 15% validación / 25% prueba

train_val_df, test_df = train_test_split(
    balanced,
    test_size=TEST_FRACTION,
    random_state=SEED,
    stratify=balanced["scientific_name"]
)

# El 15% del total equivale al 20% del bloque restante (75%).
val_fraction_of_remaining = VAL_FRACTION_TOTAL / (1.0 - TEST_FRACTION)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_fraction_of_remaining,
    random_state=SEED,
    stratify=train_val_df["scientific_name"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

split_manifest = pd.concat([train_df, val_df, test_df], ignore_index=True)
split_manifest.to_csv(SPLIT_MANIFEST_PATH, index=False)

print("Cantidades:")
print(split_manifest["split"].value_counts())
print("\nPorcentajes reales:")
print(split_manifest["split"].value_counts(normalize=True).sort_index())

print("\nDistribución por especie y split:")
display(pd.crosstab(split_manifest["scientific_name"], split_manifest["split"]))


In [ ]:
#@title 7.3 Auditoría cuantitativa del dataset

# ============================================================
# Distribución por especie y split
# ============================================================

audit_species = pd.crosstab(
    split_manifest["scientific_name"],
    split_manifest["split"]
)

# Orden deseado
for col in ["train", "val", "test"]:
    if col not in audit_species.columns:
        audit_species[col] = 0

audit_species = audit_species[
    ["train", "val", "test"]
]

audit_species["total"] = audit_species.sum(axis=1)

print("Distribución por especie:")
display(audit_species)


# ============================================================
# Distribución por condición regional
# ============================================================

audit_condition = pd.crosstab(
    split_manifest["condition_ba"],
    split_manifest["split"]
)

for col in ["train", "val", "test"]:
    if col not in audit_condition.columns:
        audit_condition[col] = 0

audit_condition = audit_condition[
    ["train", "val", "test"]
]

audit_condition["total"] = audit_condition.sum(axis=1)

print("\nDistribución por condición:")
display(audit_condition)


# ============================================================
# Resumen general
# ============================================================

dataset_summary = pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Validación", "Prueba", "Total"],
    "Cantidad": [
        len(train_df),
        len(val_df),
        len(test_df),
        len(split_manifest)
    ]
})

dataset_summary["Porcentaje"] = (
    dataset_summary["Cantidad"] /
    len(split_manifest) * 100
)

# El total debe mostrarse como 100%
dataset_summary.loc[
    dataset_summary["Conjunto"] == "Total",
    "Porcentaje"
] = 100

print("\nResumen del dataset:")
display(dataset_summary)


# ============================================================
# Validaciones automáticas
# ============================================================

print("\nControles:")

print(
    "Cantidad de especies:",
    split_manifest["scientific_name"].nunique()
)

print(
    "Cantidad de condiciones:",
    split_manifest["condition_ba"].nunique()
)

print(
    "Duplicados observation_id:",
    split_manifest["observation_id"].duplicated().sum()
)

print(
    "Duplicados local_path:",
    split_manifest["local_path"].duplicated().sum()
)


# Verifica que una observación no haya quedado en más de un split
obs_split_count = (
    split_manifest
    .groupby("observation_id")["split"]
    .nunique()
)

leakage_count = (obs_split_count > 1).sum()

print(
    "Observaciones presentes en más de un split:",
    leakage_count
)


if leakage_count == 0:
    print("✅ No se detectó leakage por observation_id.")
else:
    print("⚠️ Revisar separación del dataset.")


# ============================================================
# Guardar tablas
# ============================================================

audit_species.to_csv(
    ARTIFACTS_DIR / "auditoria_especies_split.csv"
)

audit_condition.to_csv(
    ARTIFACTS_DIR / "auditoria_condicion_split.csv"
)

dataset_summary.to_csv(
    ARTIFACTS_DIR / "auditoria_resumen_dataset.csv",
    index=False
)
if leakage_count or split_manifest["observation_id"].duplicated().any():
    raise ValueError("Observaciones duplicadas o compartidas entre particiones.")


## 8. Object Detection y recorte

Se utiliza un detector MobileNet SSD preentrenado como etapa previa de recorte. El objetivo no es clasificar la especie con el detector, sino reducir la influencia del fondo antes de entrenar el clasificador.

Limitacion esperada: el detector COCO reconoce principalmente la clase `potted plant`. Cuando no detecta una planta con suficiente confianza, el notebook conserva la imagen completa como fallback para no perder ejemplos del dataset.


In [ ]:
#@title 8.1 Cargar detector preentrenado MobileNet SSD

print("Cargando detector:", OBJECT_DETECTION_MODEL_URL)
detector = hub.load(OBJECT_DETECTION_MODEL_URL)
print("Detector cargado.")
print("Clases usadas para recorte:", DETECTION_CLASS_NAMES)


In [ ]:
#@title 8.2 Funciones de deteccion y recorte

def clamp(value, low, high):
    return max(low, min(high, value))


def expanded_pixel_box(box, width, height, padding):
    ymin, xmin, ymax, xmax = box
    x1 = int(xmin * width)
    y1 = int(ymin * height)
    x2 = int(xmax * width)
    y2 = int(ymax * height)

    bw = max(1, x2 - x1)
    bh = max(1, y2 - y1)
    pad_x = int(bw * padding)
    pad_y = int(bh * padding)

    return (
        clamp(x1 - pad_x, 0, width - 1),
        clamp(y1 - pad_y, 0, height - 1),
        clamp(x2 + pad_x, 1, width),
        clamp(y2 + pad_y, 1, height),
    )


def center_square_box(width, height):
    side = min(width, height)
    x1 = (width - side) // 2
    y1 = (height - side) // 2
    return (x1, y1, x1 + side, y1 + side)


def detect_plant_box(img):
    arr = np.asarray(img.convert("RGB"), dtype=np.uint8)
    outputs = detector(tf.convert_to_tensor(arr[None, ...], dtype=tf.uint8))

    boxes = outputs["detection_boxes"].numpy()[0]
    classes = outputs["detection_classes"].numpy()[0].astype(int)
    scores = outputs["detection_scores"].numpy()[0]

    candidates = []
    for box, cls, score in zip(boxes, classes, scores):
        if cls in DETECTION_CLASS_IDS and score >= DETECTION_SCORE_THRESHOLD:
            candidates.append((float(score), int(cls), box))

    if not candidates:
        return None

    candidates.sort(key=lambda x: x[0], reverse=True)
    score, cls, box = candidates[0]
    pixel_box = expanded_pixel_box(box, img.width, img.height, CROP_PADDING)

    return {
        "box": pixel_box,
        "score": score,
        "class_id": cls,
        "class_name": DETECTION_CLASS_NAMES.get(cls, str(cls)),
    }


def crop_image_with_detector(source_path, output_path):
    img = Image.open(source_path).convert("RGB")
    detection = detect_plant_box(img)

    if detection is not None:
        crop_box = detection["box"]
        cropped = img.crop(crop_box)
        method = "object_detection"
        score = detection["score"]
        class_id = detection["class_id"]
        class_name = detection["class_name"]
    elif FALLBACK_CENTER_CROP:
        crop_box = center_square_box(img.width, img.height)
        cropped = img.crop(crop_box)
        method = "center_crop_fallback"
        score = None
        class_id = None
        class_name = None
    else:
        crop_box = (0, 0, img.width, img.height)
        cropped = img
        method = "full_image_fallback"
        score = None
        class_id = None
        class_name = None

    output_path.parent.mkdir(parents=True, exist_ok=True)
    cropped.save(output_path, format="JPEG", quality=92)

    return {
        "crop_path": str(output_path),
        "crop_method": method,
        "detected": method == "object_detection",
        "detection_score": score,
        "detection_class_id": class_id,
        "detection_class_name": class_name,
        "crop_x1": crop_box[0],
        "crop_y1": crop_box[1],
        "crop_x2": crop_box[2],
        "crop_y2": crop_box[3],
        "source_width": img.width,
        "source_height": img.height,
    }


In [ ]:
#@title 8.3 Generar imagenes preparadas de esta corrida

def crop_split_dataset(split_manifest):
    if CROP_SPLIT_DIR.exists():
        shutil.rmtree(CROP_SPLIT_DIR)

    records = []
    for row in tqdm(split_manifest.itertuples(index=False), total=len(split_manifest)):
        source_path = Path(row.local_path)
        dst = CROP_SPLIT_DIR / row.split / row.folder_name / source_path.name
        base_record = row._asdict()

        try:
            crop_info = crop_image_with_detector(source_path, dst)
            base_record.update(crop_info)
            base_record["crop_error"] = None
            records.append(base_record)
        except Exception as e:
            base_record["crop_path"] = None
            base_record["crop_method"] = "error"
            base_record["detected"] = False
            base_record["detection_score"] = None
            base_record["detection_class_id"] = None
            base_record["detection_class_name"] = None
            base_record["crop_error"] = f"{type(e).__name__}: {str(e)[:160]}"
            records.append(base_record)

    cropped_manifest = pd.DataFrame(records)
    if cropped_manifest["crop_path"].isna().any():
        raise RuntimeError("Fallo la preparacion de imagenes. Corregir el error; no se permite descartar muestras.")
    cropped_manifest.to_csv(CROP_MANIFEST_PATH, index=False)
    print("Dataset recortado generado en:", CROP_SPLIT_DIR)
    return cropped_manifest


cropped_manifest = crop_split_dataset(split_manifest)

print("\nResumen de metodos de recorte:")
display(cropped_manifest["crop_method"].value_counts().rename("cantidad").to_frame())

crop_detection_rate = float(cropped_manifest["detected"].mean()) if len(cropped_manifest) else 0.0
print(f"Tasa de deteccion usada para recorte: {crop_detection_rate:.2%}")

cropped_manifest.to_csv(ARTIFACTS_DIR / "manifest_splits_object_detection_crop.csv", index=False)


In [ ]:
#@title 8.4 Visualizar ejemplos de recorte para el informe

sample_crop_df = (
    cropped_manifest.groupby("crop_method", group_keys=False)
    .apply(lambda x: x.sample(min(4, len(x)), random_state=SEED))
    .reset_index(drop=True)
)

if sample_crop_df.empty:
    print("No hay recortes para visualizar.")
else:
    n = min(8, len(sample_crop_df))
    sample_crop_df = sample_crop_df.head(n)

    fig, axes = plt.subplots(n, 2, figsize=(9, 3.2 * n))
    if n == 1:
        axes = np.array([axes])

    for i, (_, row) in enumerate(sample_crop_df.iterrows()):
        original = Image.open(row["local_path"]).convert("RGB")
        cropped = Image.open(row["crop_path"]).convert("RGB")

        axes[i, 0].imshow(original)
        axes[i, 0].set_title(f"Original\n{row['scientific_name']}", fontsize=9)
        axes[i, 0].axis("off")

        score_txt = "" if pd.isna(row["detection_score"]) else f" | score={row['detection_score']:.2f}"
        axes[i, 1].imshow(cropped)
        axes[i, 1].set_title(f"Recorte: {row['crop_method']}{score_txt}", fontsize=9)
        axes[i, 1].axis("off")

    plt.tight_layout()
    plt.show()
    fig.savefig(ARTIFACTS_DIR / "ejemplos_recorte_object_detection.png", dpi=160, bbox_inches="tight")

fig, ax = plt.subplots(figsize=(7, 4))
cropped_manifest["crop_method"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Metodo aplicado en la etapa de recorte")
ax.set_ylabel("Cantidad de imagenes")
ax.set_xlabel("Metodo")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()
fig.savefig(ARTIFACTS_DIR / "resumen_recortes_object_detection.png", dpi=160, bbox_inches="tight")


## 9. Carga de imagenes para clasificacion

Se cargan los recortes aceptados y las imagenes completas cuando no hubo deteccion, en RGB y redimensionadas a 224 x 224. No todas las imagenes estan recortadas.

In [ ]:
#@title 9.1 Crear datasets TensorFlow desde imagenes recortadas

train_crop_df = cropped_manifest[cropped_manifest["split"] == "train"].copy()
val_crop_df = cropped_manifest[cropped_manifest["split"] == "val"].copy()
test_crop_df = cropped_manifest[cropped_manifest["split"] == "test"].copy()

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    CROP_SPLIT_DIR / "train",
    labels="inferred",
    label_mode="categorical",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    CROP_SPLIT_DIR / "val",
    labels="inferred",
    label_mode="categorical",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    CROP_SPLIT_DIR / "test",
    labels="inferred",
    label_mode="categorical",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False
)

CLASS_NAMES = train_ds_raw.class_names
NUM_CLASSES = len(CLASS_NAMES)

print("\nClases detectadas:")
for i, c in enumerate(CLASS_NAMES):
    print(i, "->", c)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds_raw.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds_raw.cache().prefetch(AUTOTUNE)
test_ds = test_ds_raw.cache().prefetch(AUTOTUNE)

TEST_FILE_PATHS = list(test_ds_raw.file_paths)
crop_lookup = cropped_manifest.set_index("crop_path").to_dict(orient="index")


In [ ]:
#@title 9.2 Visualizar un batch de entrenamiento recortado

images_batch, labels_batch = next(iter(train_ds))

plt.figure(figsize=(12, 8))
for i in range(min(12, len(images_batch))):
    ax = plt.subplot(3, 4, i + 1)
    plt.imshow(images_batch[i].numpy().astype("uint8"))
    class_idx = int(np.argmax(labels_batch[i].numpy()))
    plt.title(CLASS_NAMES[class_idx], fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()


## 10. CNN GAP con tasa adaptable

CNN GAP desde cero, Adam con LR inicial 0,001, ReduceLROnPlateau (factor 0,5, paciencia 4, minimo 1e-5) y EarlyStopping con paciencia 12. La corrida exportada ejecuto 100 epocas y selecciono la 88. No es una ablacion controlada del recorte ni del LR: tambien cambia la seleccion y particion de datos respecto del 4.

In [ ]:
#@title 10.1 Definir y compilar CNN GAP sobre imagenes recortadas

model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_img"),
    layers.Rescaling(1./255, name="rescaling"),

    layers.Conv2D(32, 3, padding="same", activation="relu", name="conv_1"),
    layers.MaxPooling2D(name="pool_1"),

    layers.Conv2D(64, 3, padding="same", activation="relu", name="conv_2"),
    layers.MaxPooling2D(name="pool_2"),

    layers.Conv2D(128, 3, padding="same", activation="relu", name="conv_3"),
    layers.MaxPooling2D(name="pool_3"),

    layers.GlobalAveragePooling2D(name="global_average_pooling"),
    layers.Dense(128, activation="relu", name="dense_1"),
    layers.Dropout(0.30, name="dropout"),
    layers.Dense(NUM_CLASSES, activation="softmax", name="output")
], name="Exp4_ObjectDetectionCrop_CNN_GAP")

optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
#@title 10.2 Entrenar modelo

callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-5,
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

history_df = pd.DataFrame(history.history)
history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
history_df["phase"] = "training"


In [ ]:
#@title 10.3 Curvas de entrenamiento y validacion

fig, ax = plt.subplots(1, 2, figsize=(13, 4))

for phase_name, phase_df in history_df.groupby("phase", sort=False):
    ax[0].plot(phase_df["epoch"], phase_df["accuracy"], label=f"train - {phase_name}")
    ax[0].plot(phase_df["epoch"], phase_df["val_accuracy"], linestyle="--", label=f"val - {phase_name}")
    ax[1].plot(phase_df["epoch"], phase_df["loss"], label=f"train - {phase_name}")
    ax[1].plot(phase_df["epoch"], phase_df["val_loss"], linestyle="--", label=f"val - {phase_name}")

ax[0].set_title("Accuracy por epoca")
ax[0].set_xlabel("Epoca")
ax[0].set_ylabel("Accuracy")
ax[0].legend()
ax[0].grid(alpha=0.25)

ax[1].set_title("Loss por epoca")
ax[1].set_xlabel("Epoca")
ax[1].set_ylabel("Loss")
ax[1].legend()
ax[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

fig.savefig(ARTIFACTS_DIR / "curvas_entrenamiento.png", dpi=160, bbox_inches="tight")
history_df.to_csv(ARTIFACTS_DIR / "history_entrenamiento.csv", index=False)


## 11. Evaluacion del prototipo

Se evalúan **dos niveles**:

### A. Identificación de especie
Problema multiclase de 8 clases.

### B. Condición nativa/exótica
Se deriva a partir de la especie predicha.  
Esto permite separar correctamente el problema visual del conocimiento botánico regional.


In [ ]:
#@title 11.1 Evaluacion global en prueba

test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print(f"\nTest loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")


In [ ]:
#@title 11.2 Predicciones y metricas por especie

y_true_onehot = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
y_true = np.argmax(y_true_onehot, axis=1)

y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_conf = np.max(y_prob, axis=1)

species_report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

species_report_df = pd.DataFrame(species_report).T
display(species_report_df)

species_macro_f1 = f1_score(y_true, y_pred, average="macro")
species_accuracy = accuracy_score(y_true, y_pred)

print(f"Accuracy especie: {species_accuracy:.4f}")
print(f"Macro-F1 especie: {species_macro_f1:.4f}")

species_report_df.to_csv(ARTIFACTS_DIR / "classification_report_especies.csv")


In [ ]:
#@title 11.3 Matriz de confusion por especie

cm_species = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 9))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_species,
    display_labels=CLASS_NAMES
)
disp.plot(ax=ax, xticks_rotation=45, values_format="d", colorbar=False)
plt.title("Matriz de confusión — identificación de especie")
plt.tight_layout()
plt.show()

fig.savefig(ARTIFACTS_DIR / "matriz_confusion_especies.png", dpi=170, bbox_inches="tight")


In [ ]:
#@title 11.4 Evaluar clasificacion derivada Nativa / Exotica

import re
import unicodedata

def normalize_species_name(value):
    text = unicodedata.normalize("NFKD", str(value).strip().casefold())
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", "_", text).strip("_")

catalog = SPECIES_CATALOG.copy()
catalog["_species_key"] = catalog["scientific_name"].map(normalize_species_name)
if catalog["_species_key"].duplicated().any():
    raise ValueError("Especies duplicadas al normalizar el catalogo.")
catalog_by_key = catalog.set_index("_species_key")
class_to_condition = {}
for class_name in CLASS_NAMES:
    key = normalize_species_name(class_name)
    if key not in catalog_by_key.index:
        raise KeyError(f"Clase sin correspondencia regional: {class_name!r}")
    condition = catalog_by_key.loc[key, "condition_ba"]
    if condition not in ["Nativa", "Exótica"]:
        raise ValueError(f"Condicion regional invalida: {class_name!r}")
    class_to_condition[class_name] = condition

true_species_names = [CLASS_NAMES[i] for i in y_true]
pred_species_names = [CLASS_NAMES[i] for i in y_pred]

y_true_condition = np.array([class_to_condition[s] for s in true_species_names])
y_pred_condition = np.array([class_to_condition[s] for s in pred_species_names])

condition_labels = ["Nativa", "Exótica"]

condition_report = classification_report(
    y_true_condition,
    y_pred_condition,
    labels=condition_labels,
    output_dict=True,
    zero_division=0
)

condition_report_df = pd.DataFrame(condition_report).T
display(condition_report_df)

condition_accuracy = accuracy_score(y_true_condition, y_pred_condition)
condition_macro_f1 = f1_score(
    y_true_condition,
    y_pred_condition,
    labels=condition_labels,
    average="macro"
)

print(f"Accuracy condición: {condition_accuracy:.4f}")
print(f"Macro-F1 condición: {condition_macro_f1:.4f}")

condition_report_df.to_csv(ARTIFACTS_DIR / "classification_report_condicion.csv")


In [ ]:
#@title 11.5 Matriz de confusion Nativa / Exotica

cm_condition = confusion_matrix(
    y_true_condition,
    y_pred_condition,
    labels=condition_labels
)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_condition,
    display_labels=condition_labels
)
disp.plot(ax=ax, values_format="d", colorbar=False)
plt.title("Matriz de confusión — condición regional")
plt.tight_layout()
plt.show()

fig.savefig(ARTIFACTS_DIR / "matriz_confusion_condicion.png", dpi=170, bbox_inches="tight")


In [ ]:
#@title 11.6 Tabla de resultados de prueba

source_paths = []
crop_methods = []
detection_scores = []

for crop_path in TEST_FILE_PATHS:
    info = crop_lookup.get(crop_path, {})
    source_paths.append(info.get("local_path", crop_path))
    crop_methods.append(info.get("crop_method", "unknown"))
    detection_scores.append(info.get("detection_score", None))

results_test = pd.DataFrame({
    "crop_path": TEST_FILE_PATHS,
    "source_path": source_paths,
    "crop_method": crop_methods,
    "detection_score": detection_scores,
    "real_species": true_species_names,
    "pred_species": pred_species_names,
    "confidence": y_conf,
    "real_condition": y_true_condition,
    "pred_condition": y_pred_condition,
})

results_test["species_correct"] = results_test["real_species"] == results_test["pred_species"]
results_test["condition_correct"] = results_test["real_condition"] == results_test["pred_condition"]

display(results_test.head(20))

results_test.to_csv(ARTIFACTS_DIR / "resultados_test.csv", index=False)

print("\nErrores de especie:", int((~results_test["species_correct"]).sum()))
print("Errores de condicion:", int((~results_test["condition_correct"]).sum()))


## 12. Analisis de errores


In [ ]:
#@title 12.1 Mostrar errores de clasificacion mas confiados

errors_df = results_test[~results_test["species_correct"]].copy()
errors_df = errors_df.sort_values("confidence", ascending=False)

N_ERRORS_TO_SHOW = min(16, len(errors_df))

if N_ERRORS_TO_SHOW == 0:
    print("No hubo errores de especie en el conjunto de prueba.")
else:
    cols = 4
    rows = int(np.ceil(N_ERRORS_TO_SHOW / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.5 * rows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, errors_df.head(N_ERRORS_TO_SHOW).iterrows()):
        img = Image.open(row["crop_path"])
        ax.imshow(img)
        ax.set_title(
            f"Real: {row['real_species']}\n"
            f"Pred: {row['pred_species']}\n"
            f"Conf.: {row['confidence']:.1%}",
            fontsize=8
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    fig.savefig(ARTIFACTS_DIR / "errores_representativos.png", dpi=160, bbox_inches="tight")


In [ ]:
#@title 12.2 Resumen de pares de confusion

if len(errors_df):
    confusion_pairs = (
        errors_df.groupby(["real_species", "pred_species"])
                 .size()
                 .rename("cantidad")
                 .reset_index()
                 .sort_values("cantidad", ascending=False)
    )
    display(confusion_pairs.head(20))
else:
    print("Sin pares de confusión para mostrar.")


In [ ]:
#@title 12.3 Mostrar 2 aciertos de clasificacion por especie

N_CORRECT_PER_SPECIES = 2
correct_df = results_test[results_test["species_correct"]].copy()
correct_df = correct_df.sort_values("confidence", ascending=False)

selected_examples = []
for species in CLASS_NAMES:
    species_df = correct_df[correct_df["real_species"] == species].head(N_CORRECT_PER_SPECIES)
    selected_examples.append(species_df)

selected_df = pd.concat(selected_examples, ignore_index=True) if selected_examples else pd.DataFrame()

if selected_df.empty:
    print("No hubo aciertos de especie en el conjunto de prueba.")
else:
    cols = 4
    rows = int(np.ceil(len(selected_df) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.5 * rows))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, selected_df.iterrows()):
        img = Image.open(row["crop_path"])
        ax.imshow(img)
        ax.set_title(
            f"Especie: {row['real_species']}\n"
            f"Pred: {row['pred_species']}\n"
            f"Conf.: {row['confidence']:.1%}",
            fontsize=8
        )
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    fig.savefig(ARTIFACTS_DIR / "aciertos_2_por_especie.png", dpi=160, bbox_inches="tight")


## 13. Exportacion directa a runs

La celda 3 crea la corrida y todos sus resultados se guardan directamente en `artifacts_entrega3/runs`, sin duplicarlos en una carpeta por experimento. Las imagenes preparadas quedan en `/content/iaa_work` y se pierden al terminar la sesion. Se conservan metricas, predicciones, reportes, auditorias, manifiestos, curvas, matrices y ejemplos para el informe. El guardado `.keras` sigue comentado por tamano. Repetir la celda 3 crea otra corrida; repetir solo la exportacion actualiza la actual.

Los archivos y las salidas historicas conservan las rutas de la ejecucion original. La correccion de codigo no regenera ni altera esos resultados.


In [ ]:
#@title 13.1 Exportar metricas, tablas e imagenes EN GOOGLE DRIVE

best_val_loss_epoch = None
if "val_loss" in history_df.columns and len(history_df):
    best_val_loss_epoch = int(history_df.loc[history_df["val_loss"].idxmin(), "epoch"])

trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
non_trainable_params = int(np.sum([np.prod(v.shape) for v in model.non_trainable_weights]))
total_params = trainable_params + non_trainable_params

metrics_summary = {
    "run_name": RUN_NAME,
    "dataset_partition_sha256": dataset_partition_sha256(cropped_manifest),
    "timestamp": RUN_TIMESTAMP,
    "experiment_number": int(EXPERIMENT_NUMBER),
    "experiment_id": EXPERIMENT_ID,
    "experiment_title": EXPERIMENT_TITLE,
    "seed": int(SEED),
    "img_size": int(IMG_SIZE),
    "batch_size": int(BATCH_SIZE),
    "learning_rate": float(LEARNING_RATE),
    "epochs_configured": int(EPOCHS),
    "epochs_executed": int(len(history_df)),
    "best_val_loss_epoch": best_val_loss_epoch,
    "n_species": int(NUM_CLASSES),
    "n_train": int(len(train_crop_df)),
    "n_val": int(len(val_crop_df)),
    "n_test": int(len(test_crop_df)),
    "object_detection_enabled": True,
    "object_detection_model_url": OBJECT_DETECTION_MODEL_URL,
    "detection_class_ids": [int(x) for x in DETECTION_CLASS_IDS],
    "detection_score_threshold": float(DETECTION_SCORE_THRESHOLD),
    "crop_padding": float(CROP_PADDING),
    "crop_detection_rate": float(crop_detection_rate),
    "crop_methods": cropped_manifest["crop_method"].value_counts().to_dict(),
    "base_classifier": "CNN_GAP_desde_cero",
    "transfer_learning": False,
    "fine_tuning": False,
    "trainable_params": trainable_params,
    "non_trainable_params": non_trainable_params,
    "total_params": total_params,
    "test_loss": float(test_loss),
    "test_accuracy_metric": float(test_accuracy),
    "species_accuracy": float(species_accuracy),
    "species_macro_f1": float(species_macro_f1),
    "condition_accuracy": float(condition_accuracy),
    "condition_macro_f1": float(condition_macro_f1),
    "classes": list(CLASS_NAMES),
}

with open(RUN_DIR / f"metricas_entrega3_exp{EXPERIMENT_NUMBER}.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)

history_df.to_csv(RUN_DIR / "history_entrenamiento.csv", index=False)
results_test.to_csv(RUN_DIR / "resultados_test.csv", index=False)
species_report_df.to_csv(RUN_DIR / "classification_report_especies.csv")
condition_report_df.to_csv(RUN_DIR / "classification_report_condicion.csv")
cropped_manifest.to_csv(RUN_DIR / "manifest_splits_object_detection_crop.csv", index=False)

with open(RUN_DIR / "model_summary.txt", "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda x: f.write(x + "\n"))

# El modelo .keras no se exporta por tamano. Descomentar solo si se necesita guardar pesos/modelo.
# MODEL_PATH = RUN_DIR / f"modelo_exp{EXPERIMENT_NUMBER}.keras"
# model.save(str(MODEL_PATH))

README_PATH = RUN_DIR / "README.txt"
with open(README_PATH, "w", encoding="utf-8") as f:
    f.write("IAA - Entrega 3 - Identificacion de especies vegetales\n")
    f.write("=" * 64 + "\n\n")
    f.write(f"Experimento: {EXPERIMENT_NUMBER} - {EXPERIMENT_TITLE}\n")
    f.write(f"Run: {RUN_NAME}\n")
    f.write(f"Resolucion: {IMG_SIZE}x{IMG_SIZE}\n")
    f.write(f"Batch size: {BATCH_SIZE}\n")
    f.write(f"Learning rate: {LEARNING_RATE}\n")
    f.write(f"Epocas ejecutadas: {len(history_df)}\n")
    f.write(f"Mejor epoca por val_loss: {best_val_loss_epoch}\n\n")
    f.write(f"Accuracy especie: {species_accuracy:.4f}\n")
    f.write(f"Macro-F1 especie: {species_macro_f1:.4f}\n")
    f.write(f"Accuracy condicion: {condition_accuracy:.4f}\n")
    f.write(f"Macro-F1 condicion: {condition_macro_f1:.4f}\n")
    f.write(f"Test loss: {test_loss:.4f}\n\n")
    f.write(f"Tasa de deteccion para recorte: {crop_detection_rate:.2%}\n")
    f.write("Nota: el modelo .keras no se exporta automaticamente por tamano.\n")

print("\nArchivos exportados en:")
print(RUN_DIR)
print("\nResumen:")
print(json.dumps(metrics_summary, ensure_ascii=False, indent=2))


## 14. Inferencia manual

Funcion opcional para probar una imagen externa usando el mismo pipeline de recorte y clasificacion.


In [ ]:
#@title 14.1 Funcion de inferencia con recorte + Top-3

def predict_plant_image(image_path, show=True, top_k=3):
    temp_dir = ARTIFACTS_DIR / "tmp_inference"
    temp_dir.mkdir(parents=True, exist_ok=True)
    crop_path = temp_dir / f"crop_{Path(image_path).name}"

    crop_info = crop_image_with_detector(Path(image_path), crop_path)
    img_crop = Image.open(crop_path).convert("RGB")
    img_resized = img_crop.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)

    x = np.array(img_resized, dtype=np.float32)
    x = np.expand_dims(x, axis=0)

    probs = model.predict(x, verbose=0)[0]
    top_indices = np.argsort(probs)[::-1][:top_k]

    top_predictions = []
    for rank, idx in enumerate(top_indices, start=1):
        folder_species = CLASS_NAMES[idx]
        catalog_row = catalog_by_key.loc[normalize_species_name(folder_species)]
        top_predictions.append({
            "rank": rank,
            "species": catalog_row["scientific_name"],
            "common_name": catalog_row["common_name"],
            "condition_ba": catalog_row["condition_ba"],
            "softmax_probability": float(probs[idx])
        })

    result = top_predictions[0]

    if show:
        original = Image.open(image_path).convert("RGB")
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(original)
        axes[0].set_title("Original")
        axes[0].axis("off")
        axes[1].imshow(img_crop)
        axes[1].set_title(f"Recorte: {crop_info['crop_method']}")
        axes[1].axis("off")
        plt.suptitle(
            f"{result['species']} ({result['common_name']})\n"
            f"{result['condition_ba']} | Prob. Softmax: {result['softmax_probability']:.1%}"
        )
        plt.tight_layout()
        plt.show()

        print("TOP", top_k, "PREDICCIONES\n")
        for pred in top_predictions:
            print(f"{pred['rank']}. {pred['species']} ({pred['common_name']})")
            print(f"   Condicion: {pred['condition_ba']}")
            print(f"   Probabilidad Softmax: {pred['softmax_probability']:.2%}")

    return {
        "prediction": result,
        "top_predictions": top_predictions,
        "crop_info": crop_info,
    }
